# Netflix Tableau Data Prep

In [ ]:
"""
Netflix EDA prep for Tableau  (lean version)

Usage:
    python netflix_prep.py                       # downloads the CSV from the course link
    python netflix_prep.py netflix.csv out_dir   # local file + output folder

Outputs (CSV, one table per grain; relate them in Tableau on show_id):
    netflix_titles.csv         1 row per title  (typed columns + derived columns)
    netflix_cast.csv           title x actor    (is_lead = first listed)
    netflix_director.csv       title x director
    netflix_country.csv        title x country
    netflix_genre.csv          title x genre
    netflix_director_actor.csv title x director x actor (network analysis)
"""
import sys
from pathlib import Path

import pandas as pd

PATH = "assets/netflix.csv"

# ---------------------------------------------------------------- 1. load
df = pd.read_csv(PATH)
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]          # stray pandas index column

# ---------------------------------------------------------------- 2. tidy text
text_cols = df.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    df[c] = df[c].str.strip().replace("", pd.NA)

# Runtime typed into `rating` with `duration` blank (a few Louis C.K. specials)
shifted = df["duration"].isna() & df["rating"].str.fullmatch(r"\d+ min", na=False)
df.loc[shifted, ["duration", "rating"]] = df.loc[shifted, ["rating", "duration"]].values
print(f"rows with runtime in rating column, fixed: {shifted.sum()}")

# Duplicate checks (report only, nothing deleted)
print("duplicate show_id:", df["show_id"].duplicated().sum())
print("duplicate title/type/year:", df.duplicated(["title", "type", "release_year"]).sum())

# Missing-value profile (needed for the report)
print("\nmissing share per column:\n", df.isna().mean().round(3).to_string())

# ---------------------------------------------------------------- 3. bridge tables
def unnest(frame, col, name):
    """One row per title per comma-separated value; pos = listed order."""
    s = frame[["show_id", col]].dropna()
    s = s.assign(**{name: s[col].str.split(",")}).explode(name)
    s[name] = s[name].str.strip()
    s = s[s[name] != ""].drop_duplicates(["show_id", name])
    s["pos"] = s.groupby("show_id").cumcount() + 1
    return s[["show_id", "pos", name]].reset_index(drop=True)

cast = unnest(df, "cast", "actor")
cast["is_lead"] = (cast["pos"] == 1).astype(int)
director = unnest(df, "director", "director")
genre = unnest(df, "listed_in", "genre")

country = unnest(df, "country", "country")
# defunct names Tableau cannot place on a map
country["country"] = country["country"].replace({"West Germany": "Germany",
                                                 "Soviet Union": "Russia"})
# keep titles with no country so counts reconcile
no_country = df.loc[~df["show_id"].isin(country["show_id"]), "show_id"]
country = pd.concat(
    [country, pd.DataFrame({"show_id": no_country, "pos": 1, "country": "Unknown"})],
    ignore_index=True)

# ---------------------------------------------------------------- 4. title table
t = pd.DataFrame({
    "show_id": df["show_id"],
    "content_type": df["type"],
    "title": df["title"],
    "description": df["description"],
    "release_year": df["release_year"].astype(int),
})
t["release_decade"] = t["release_year"] // 10 * 10

# date added
d = pd.to_datetime(df["date_added"], format="%B %d, %Y", errors="coerce")
t["date_added"] = d
t["year_added"] = d.dt.year.astype("Int64")
t["month_added"] = d.dt.month.astype("Int64")
t["month_name"] = d.dt.month_name()
t["quarter_added"] = d.dt.quarter.astype("Int64")
t["years_release_to_netflix"] = t["year_added"] - t["release_year"]
# data ends 2021-09-25, so 2021 is a partial year
t["is_complete_year_added"] = (t["year_added"] < 2021).astype("Int64").where(d.notna())

# rating
r = df["rating"].fillna("Unknown").replace({"NR": "Not Rated", "UR": "Not Rated"})
t["rating"] = r

# duration -> two numeric columns
t["duration_raw"] = df["duration"]
t["movie_minutes"] = (df["duration"].str.extract(r"^(\d+) min$")[0]
                      .astype("Int64").where(df["type"] == "Movie"))
t["tv_seasons"] = (df["duration"].str.extract(r"^(\d+) Season")[0]
                   .astype("Int64").where(df["type"] == "TV Show"))

# per-title counts and first-listed values from the bridge tables
real_country = country[country["country"] != "Unknown"]
t["country_count"] = t["show_id"].map(real_country.groupby("show_id").size()).fillna(0).astype(int)
t["is_co_production"] = (t["country_count"] > 1).astype(int)
t["primary_country"] = t["show_id"].map(country[country["pos"] == 1].set_index("show_id")["country"]).fillna("Unknown")
t["primary_genre"] = t["show_id"].map(genre[genre["pos"] == 1].set_index("show_id")["genre"]).fillna("Unknown")
t["genre_count"] = t["show_id"].map(genre.groupby("show_id").size()).fillna(0).astype(int)
t["cast_count"] = t["show_id"].map(cast.groupby("show_id").size()).fillna(0).astype(int)
t["director_count"] = t["show_id"].map(director.groupby("show_id").size()).fillna(0).astype(int)
t["has_cast"] = (t["cast_count"] > 0).astype(int)
t["has_director"] = (t["director_count"] > 0).astype(int)

# replaces the Tableau calculated fields
t["title_count"] = 1                                   # SUM(title_count) = number of titles
known = t["primary_country"] != "Unknown"
t["is_non_us"] = (t["primary_country"] != "United States").astype("Int64").where(known)  # NULL if country unknown

# category dtype (rubric asks for it; no effect on the CSVs)
for c in ["content_type", "rating", "primary_country", "primary_genre"]:
    t[c] = t[c].astype("category")

# ---------------------------------------------------------------- 5. network table
director_actor = (director[["show_id", "director"]]
                  .merge(cast[["show_id", "actor", "is_lead"]], on="show_id")
                  .merge(t[["show_id", "content_type", "primary_country"]], on="show_id"))
director_actor["pair"] = director_actor["director"] + " & " + director_actor["actor"]   # label for the pairs chart

# ---------------------------------------------------------------- 6. checks + export
print("\ntitles:", len(t), "| distinct ids:", t["show_id"].nunique())
print(t["content_type"].value_counts().to_string())
print("date range:", t["date_added"].min().date(), "->", t["date_added"].max().date())
print("max cast size:", t["cast_count"].max())

tables = {"netflix_titles": t, "netflix_cast": cast, "netflix_director": director,
          "netflix_country": country, "netflix_genre": genre,
          "netflix_director_actor": director_actor}
for name, frame in tables.items():
    frame.to_csv(f"assets/tableau/tableau_dataset/{name}.csv", index=False, encoding="utf-8")
    print(f"wrote {name}.csv  ({len(frame):,} rows)")

rows with runtime in rating column, fixed: 3
duplicate show_id: 0
duplicate title/type/year: 1

missing share per column:
 show_id         0.000
type            0.000
title           0.000
director        0.299
cast            0.094
country         0.094
date_added      0.001
release_year    0.000
rating          0.001
duration        0.000
listed_in       0.000
description     0.000

titles: 8807 | distinct ids: 8807
content_type
Movie      6131
TV Show    2676
date range: 2008-01-01 -> 2021-09-25
max cast size: 50
wrote netflix_titles.csv  (8,807 rows)
wrote netflix_cast.csv  (64,124 rows)
wrote netflix_director.csv  (6,977 rows)
wrote netflix_country.csv  (10,843 rows)
wrote netflix_genre.csv  (19,323 rows)
wrote netflix_director_actor.csv  (51,181 rows)
